In [1]:
!pip install openpyxl

# Bibliotecas nativas
import os
import xml.etree.ElementTree as ET
from decimal import Decimal, InvalidOperation

# Bibliotecas externas
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

In [6]:
import os
import xml.etree.ElementTree as ET

os.makedirs("notas_xml", exist_ok=True)

notas_amostra = [
    {"num": "1001", "data": "2026-10-10",
     "fornecedor": "Tech Solutions LTDA",
     "cnpj": "12.345.678/0001-90", "total": "4500.00"},

    {"num": "1002", "data": "2026-10-12",
     "fornecedor": "Cloud Cloud Servicos",
     "cnpj": "98.765.432/0001-11", "total": "1280.50"},

    {"num": "1003", "data": "2026-10-15",
     "fornecedor": "Hardware & Redes SA",
     "cnpj": "45.123.890/0001-55", "total": "8900.00"}
]

for n in notas_amostra:

    raiz = ET.Element("NFe")
    inf = ET.SubElement(raiz, "infNFe")

    ide = ET.SubElement(inf, "ide")
    ET.SubElement(ide, "nNF").text = n["num"]
    ET.SubElement(ide, "dhEmi").text = (
        n["data"] + "T14:30:00-03:00"
    )

    emit = ET.SubElement(inf, "emit")
    ET.SubElement(emit, "CNPJ").text = n["cnpj"]
    ET.SubElement(emit, "xNome").text = n["fornecedor"]

    total = ET.SubElement(inf, "total")
    icms = ET.SubElement(total, "ICMSTot")
    ET.SubElement(icms, "vNF").text = n["total"]

    arvore = ET.ElementTree(raiz)

    arvore.write(
        f"notas_xml/nfe_{n['num']}.xml",
        encoding="utf-8",
        xml_declaration=True
    )

print("Os três arquivos XML foram criados corretamente!")

Os três arquivos XML foram criados corretamente!


In [7]:
dados_notas = []

for arquivo in os.listdir("notas_xml"):

    if arquivo.endswith(".xml"):

        caminho = os.path.join("notas_xml", arquivo)

        try:
            arvore = ET.parse(caminho)
            raiz = arvore.getroot()

            numero = raiz.findtext(".//nNF")
            data = raiz.findtext(".//dhEmi")
            fornecedor = raiz.findtext(".//xNome")
            cnpj = raiz.findtext(".//CNPJ")
            valor = raiz.findtext(".//vNF")

            if not all([numero, data, fornecedor, cnpj, valor]):
                raise ValueError("Dados ausentes no XML")

            dados_notas.append({
                "Número": numero,
                "Data": data.split("T")[0],
                "Fornecedor": fornecedor,
                "CNPJ": cnpj,
                "Valor": Decimal(valor)
            })

            print(f"{arquivo} lido com sucesso!")

        except (ET.ParseError, OSError, ValueError,
                InvalidOperation) as erro:
            print(f"Erro em {arquivo}: {erro}")

print("\nTotal de notas:", len(dados_notas))

nfe_1003.xml lido com sucesso!
nfe_1001.xml lido com sucesso!
nfe_1002.xml lido com sucesso!

Total de notas: 3


In [8]:
workbook = Workbook()
planilha = workbook.active
planilha.title = "Relatório Fiscal"

planilha.append([
    "Número da Nota",
    "Data de Emissão",
    "Fornecedor",
    "CNPJ",
    "Valor Total"
])

# Formatar cabeçalho
for celula in planilha[1]:
    celula.font = Font(bold=True, color="FFFFFF")
    celula.fill = PatternFill(
        start_color="1F4E78",
        end_color="1F4E78",
        fill_type="solid"
    )

# Inserir dados
for nota in dados_notas:
    planilha.append([
        nota["Número"],
        nota["Data"],
        nota["Fornecedor"],
        nota["CNPJ"],
        float(nota["Valor"])
    ])

# Formatar valores
for linha in range(2, planilha.max_row + 1):
    planilha.cell(linha, 5).number_format = '"R$" #,##0.00'

# Ajustar colunas
for coluna, largura in {
    "A": 18, "B": 20, "C": 30, "D": 22, "E": 20
}.items():
    planilha.column_dimensions[coluna].width = largura

# Salvar planilha
workbook.save("relatorio_fiscal.xlsx")

print("Planilha criada com sucesso!")

Planilha criada com sucesso!


In [9]:
from google.colab import files

files.download("relatorio_fiscal.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>